In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import time

import numpy as np
from PIL import Image

import ot
from sklearn.neighbors import KNeighborsRegressor
from scipy.ndimage import map_coordinates

import torch


try:
    from DISTS_pytorch import DISTS
except ImportError as e:
    raise ImportError("Missing DISTS. Run in a notebook cell: pip install DISTS-pytorch") from e

In [2]:
pip install DISTS-pytorch

Note: you may need to restart the kernel to use updated packages.


In [3]:
def load_image_rgb01(path: str | Path):
    img = Image.open(path).convert("RGB")
    arr = np.asarray(img, dtype=np.float32) / 255.0
    return img, arr  

def flatten_rgb(img_rgb01: np.ndarray):
    return img_rgb01.reshape(-1, 3).astype(np.float32)

def uniform_weights(n: int):
    return (np.ones((n,), dtype=np.float32) / n)

def sample_points(X: np.ndarray, Y: np.ndarray, n_sample: int, seed: int = 0):
    rng = np.random.default_rng(seed)
    ns = min(n_sample, X.shape[0], Y.shape[0])
    idx_X = rng.choice(X.shape[0], ns, replace=False)
    idx_Y = rng.choice(Y.shape[0], ns, replace=False)
    return idx_X, idx_Y, X[idx_X].astype(np.float32), Y[idx_Y].astype(np.float32)

def resize_to(img_np_rgb01: np.ndarray, H: int, W: int):
    im = Image.fromarray((np.clip(img_np_rgb01, 0, 1) * 255).astype(np.uint8))
    im = im.resize((W, H), resample=Image.BICUBIC)
    return np.asarray(im, dtype=np.float32) / 255.0

def dists_score(dists_model, img_a_np, img_b_np, device):

    a = torch.from_numpy(img_a_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0).to(device)
    b = torch.from_numpy(img_b_np.astype(np.float32)).permute(2, 0, 1).unsqueeze(0).to(device)
    with torch.no_grad():
        return float(dists_model(a, b).item())

In [4]:
def sym_power(M: np.ndarray, power: float, reg: float = 1e-8):
    M = (M + M.T) * 0.5 + reg * np.eye(M.shape[0], dtype=np.float64)
    evals, evecs = np.linalg.eigh(M)
    evals = np.maximum(evals, reg)
    return (evecs @ np.diag(evals ** power) @ evecs.T)

def precond_tabak_fit(X: np.ndarray, Y: np.ndarray, reg: float = 1e-6):
    """
    Returns (mX, mY, A) s.t. X_aligned = (X - mX) @ A.T + mY
    """
    X64 = X.astype(np.float64)
    Y64 = Y.astype(np.float64)
    mX = X64.mean(axis=0)
    mY = Y64.mean(axis=0)
    Cx = np.cov(X64 - mX, rowvar=False) + reg * np.eye(3)
    Cy = np.cov(Y64 - mY, rowvar=False) + reg * np.eye(3)

    Cy_half = sym_power(Cy, 0.5, reg=reg)
    inner = Cy_half @ Cx @ Cy_half
    inner_inv_half = sym_power(inner, -0.5, reg=reg)
    A = Cy_half @ inner_inv_half @ Cy_half

    return mX.astype(np.float32), mY.astype(np.float32), A.astype(np.float32)

def precond_tabak_apply(X: np.ndarray, mX: np.ndarray, mY: np.ndarray, A: np.ndarray):
    return ((X - mX[None, :]) @ A.T) + mY[None, :]

In [5]:
def sinkhorn_plain(C, a, b, eps=0.05, n_iters=500, tol=1e-3, log_every=10):
    K = np.exp(-C / eps).astype(np.float32)
    u = np.ones_like(a, dtype=np.float32)
    v = np.ones_like(b, dtype=np.float32)

    logs = []
    for it in range(n_iters):
        Kv = K @ v + 1e-32
        u = a / Kv

        KTu = K.T @ u + 1e-32
        v = b / KTu

        m_u = u * (K @ v + 1e-32)
        m_v = v * (K.T @ u + 1e-32)
        err_u = float(np.max(np.abs(m_u - a)))
        err_v = float(np.max(np.abs(m_v - b)))
        err = max(err_u, err_v)

        if it % log_every == 0:
            logs.append(f"[iter {it:4d}] err={err:.3e}  err_u={err_u:.3e}  err_v={err_v:.3e}")
        if err < tol:
            logs.append(f"Converged at iter {it}, err={err:.3e}")
            break

    return u, v, it, logs


def sinkhorn_variable_damped(
    C, a, b,
    eps=0.05,
    n_iters=500,
    alpha=0.5,
    theta_min=0.6,
    theta_max=1.0,
    tol=1e-3,
    log_every=10,
):
    K = np.exp(-C / eps).astype(np.float32)
    u = np.ones_like(a, dtype=np.float32)
    v = np.ones_like(b, dtype=np.float32)

    logs = []
    for it in range(n_iters):
        Kv = K @ v + 1e-32
        m_u = u * Kv
        s_u = alpha * a + (1 - alpha) * m_u
        theta_u = np.clip(a / s_u, theta_min, theta_max)

        u_new = a / Kv
        u = u ** (1 - theta_u) * u_new ** theta_u

        KTu = K.T @ u + 1e-32
        m_v = v * KTu
        s_v = alpha * b + (1 - alpha) * m_v
        theta_v = np.clip(b / s_v, theta_min, theta_max)

        v_new = b / KTu
        v = v ** (1 - theta_v) * v_new ** theta_v

        err_u = float(np.max(np.abs(m_u - a)))
        err_v = float(np.max(np.abs(m_v - b)))
        err = max(err_u, err_v)

        if it % log_every == 0:
            logs.append(f"[iter {it:4d}] err={err:.3e}  err_u={err_u:.3e}  err_v={err_v:.3e}")
        if err < tol:
            logs.append(f"Converged at iter {it}, err={err:.3e}")
            break

    return u, v, it, logs

In [6]:
def apply_3d_lut(X_img_rgb01: np.ndarray, X_samples: np.ndarray, X_transported: np.ndarray,
                 grid_res: int = 33, knn_k: int = 5):
    grid_1d = np.linspace(0, 1, grid_res).astype(np.float32)
    grid_points = np.stack(np.meshgrid(grid_1d, grid_1d, grid_1d, indexing="ij"), axis=-1).reshape(-1, 3)

    knn = KNeighborsRegressor(n_neighbors=knn_k, weights="distance")
    knn.fit(X_samples.astype(np.float32), X_transported.astype(np.float32))
    lut = knn.predict(grid_points).reshape(grid_res, grid_res, grid_res, 3).astype(np.float32)

    coords = flatten_rgb(X_img_rgb01).T * (grid_res - 1)  
    lut_channels = lut.transpose(3, 0, 1, 2)            

    out = np.zeros_like(coords, dtype=np.float32)
    for c in range(3):
        out[c] = map_coordinates(lut_channels[c], coords, order=1, mode="nearest")

    out_img = out.T.reshape(X_img_rgb01.shape)
    return np.clip(out_img, 0.0, 1.0)

In [7]:
def ot_barycentric_emd(Xs: np.ndarray, Ys: np.ndarray):
    ns = Xs.shape[0]
    a = uniform_weights(ns)
    b = uniform_weights(ns)
    C = ot.dist(Xs, Ys, metric="sqeuclidean").astype(np.float32)
    t0 = time.perf_counter()
    G = ot.emd(a, b, C).astype(np.float32)
    Y_bar = (G @ Ys) / (G.sum(axis=1, keepdims=True) + 1e-32)
    t1 = time.perf_counter()
    return Y_bar.astype(np.float32), {"solver": "emd", "ot_time_sec": (t1 - t0), "iters": None, "logs": []}

def ot_barycentric_pot_sinkhorn(Xs: np.ndarray, Ys: np.ndarray, reg: float):
    ns = Xs.shape[0]
    a = uniform_weights(ns)
    b = uniform_weights(ns)
    C = ot.dist(Xs, Ys, metric="sqeuclidean").astype(np.float32)
    Cn = C / (C.max() + 1e-32)
    t0 = time.perf_counter()
    G = ot.sinkhorn(a, b, Cn, reg=reg).astype(np.float32)
    Y_bar = (G @ Ys) / (G.sum(axis=1, keepdims=True) + 1e-32)
    t1 = time.perf_counter()
    return Y_bar.astype(np.float32), {"solver": "pot_sinkhorn", "ot_time_sec": (t1 - t0), "iters": None, "logs": []}

def ot_barycentric_plain_sinkhorn(Xs: np.ndarray, Ys: np.ndarray, eps: float, n_iters: int, tol: float, log_every: int):
    ns = Xs.shape[0]
    a = uniform_weights(ns)
    b = uniform_weights(ns)
    C = ot.dist(Xs, Ys, metric="euclidean").astype(np.float32)

    t0 = time.perf_counter()
    u, v, it, logs = sinkhorn_plain(C, a, b, eps=eps, n_iters=n_iters, tol=tol, log_every=log_every)
    K = np.exp(-C / eps).astype(np.float32)
    P = (u[:, None] * K) * v[None, :]
    Y_bar = (P @ Ys) / (P.sum(axis=1, keepdims=True) + 1e-32)
    t1 = time.perf_counter()

    return Y_bar.astype(np.float32), {"solver": "plain_sinkhorn", "ot_time_sec": (t1 - t0), "iters": int(it), "logs": logs}

def ot_barycentric_variable_sinkhorn(Xs: np.ndarray, Ys: np.ndarray, eps: float, n_iters: int, tol: float, log_every: int,
                                     alpha: float, theta_min: float, theta_max: float):
    ns = Xs.shape[0]
    a = uniform_weights(ns)
    b = uniform_weights(ns)
    C = ot.dist(Xs, Ys, metric="euclidean").astype(np.float32)

    t0 = time.perf_counter()
    u, v, it, logs = sinkhorn_variable_damped(
        C, a, b, eps=eps, n_iters=n_iters, tol=tol, log_every=log_every,
        alpha=alpha, theta_min=theta_min, theta_max=theta_max
    )
    K = np.exp(-C / eps).astype(np.float32)
    P = (u[:, None] * K) * v[None, :]
    Y_bar = (P @ Ys) / (P.sum(axis=1, keepdims=True) + 1e-32)
    t1 = time.perf_counter()

    return Y_bar.astype(np.float32), {"solver": "variable_sinkhorn", "ot_time_sec": (t1 - t0), "iters": int(it), "logs": logs}

In [8]:
@dataclass
class TransferConfig:
    n_sample: int = 20000
    seed: int = 0

    # plain / variable params
    eps: float = 0.03
    n_iters: int = 500
    tol: float = 1e-3
    log_every: int = 10

    # variable damping
    alpha: float = 0.5
    theta_min: float = 0.6
    theta_max: float = 1.0

    # POT sinkhorn reg
    pot_reg: float = 0.01

    # LUT
    knn_k: int = 5
    grid_res: int = 33

    # timing scope
    include_deploy_in_time: bool = False  # True = end-to-end (OT + LUT)


METHODS_8 = [
    "precond_emd_lut",
    "precond_pot_sinkhorn_lut",
    "precond_plain_sinkhorn_lut",
    "precond_variable_sinkhorn_lut",
    "raw_emd_lut",
    "raw_pot_sinkhorn_lut",
    "raw_plain_sinkhorn_lut",
    "raw_variable_sinkhorn_lut",
]


def transfer_color_lut(src_path: str | Path, tgt_path: str | Path, method: str, cfg: TransferConfig):
    if method not in METHODS_8:
        raise ValueError(f"Unknown method '{method}'. Choose from: {METHODS_8}")

    _, src_img = load_image_rgb01(src_path)
    _, tgt_img = load_image_rgb01(tgt_path)

    X_full = flatten_rgb(src_img)
    Y_full = flatten_rgb(tgt_img)

    idx_X, idx_Y, Xs_rgb, Ys_rgb = sample_points(X_full, Y_full, cfg.n_sample, seed=cfg.seed)

    use_precond = method.startswith("precond_")
    if use_precond:
        mX, mY, A = precond_tabak_fit(Xs_rgb, Ys_rgb)
        Xs = precond_tabak_apply(Xs_rgb, mX, mY, A).astype(np.float32)
        Ys = Ys_rgb.astype(np.float32)
    else:
        Xs = Xs_rgb.astype(np.float32)
        Ys = Ys_rgb.astype(np.float32)

    #  OT timing 

    t_all0 = time.perf_counter()

    if method.endswith("emd_lut"):
        Y_bar, meta = ot_barycentric_emd(Xs, Ys)
    elif method.endswith("pot_sinkhorn_lut"):
        Y_bar, meta = ot_barycentric_pot_sinkhorn(Xs, Ys, reg=cfg.pot_reg)
    elif method.endswith("plain_sinkhorn_lut"):
        Y_bar, meta = ot_barycentric_plain_sinkhorn(Xs, Ys, eps=cfg.eps, n_iters=cfg.n_iters, tol=cfg.tol, log_every=cfg.log_every)
    elif method.endswith("variable_sinkhorn_lut"):
        Y_bar, meta = ot_barycentric_variable_sinkhorn(
            Xs, Ys, eps=cfg.eps, n_iters=cfg.n_iters, tol=cfg.tol, log_every=cfg.log_every,
            alpha=cfg.alpha, theta_min=cfg.theta_min, theta_max=cfg.theta_max
        )
    else:
        raise RuntimeError("Unreachable branch")


    Y_mapped_rgb = Y_bar.astype(np.float32)

    out_img = apply_3d_lut(
        X_img_rgb01=src_img,
        X_samples=Xs_rgb,          # features: original source RGB samples
        X_transported=Y_mapped_rgb,
        grid_res=cfg.grid_res,
        knn_k=cfg.knn_k
    )

    t_all1 = time.perf_counter()


    if cfg.include_deploy_in_time:
        meta["time_sec_reported"] = float(t_all1 - t_all0)
        meta["time_scope"] = "end_to_end_ot_plus_lut"
    else:
        meta["time_sec_reported"] = float(meta["ot_time_sec"])
        meta["time_scope"] = "ot_only"

    meta.update({
        "method": method,
        "use_precond": bool(use_precond),
        "n_sample_effective": int(min(cfg.n_sample, X_full.shape[0], Y_full.shape[0])),
    })

    out_pil = Image.fromarray((out_img * 255).astype(np.uint8))
    return out_pil, out_img.astype(np.float32), tgt_img, meta

In [22]:
import pandas as pd

# Paths
src_path = "penacony.png"
tgt_path = "sunset.jpg"

# Config
cfg = TransferConfig(
    n_sample=5000,
    seed=0,
    eps=0.03,
    n_iters=500,
    tol=1e-3,
    pot_reg=0.01,
    knn_k=5,
    grid_res=33,
    include_deploy_in_time=False,   
)

# DISTS model
device = "cuda" if torch.cuda.is_available() else "cpu"
dists_model = DISTS().to(device).eval()

rows = []
outputs = {}  # store PIL outputs for display/save

for method in METHODS_8:
    out_pil, out_np, tgt_img_np, meta = transfer_color_lut(src_path, tgt_path, method, cfg)

    # resize target to output size for fair DISTS
    H, W = out_np.shape[0], out_np.shape[1]
    tgt_rs = resize_to(tgt_img_np, H, W)

    d = dists_score(dists_model, out_np, tgt_rs, device=device)

    row = {
        "method": method,
        "use_precond": meta["use_precond"],
        "solver": meta["solver"],
        "time_scope": meta["time_scope"],
        "time_sec": meta["time_sec_reported"],
        "sinkhorn_iters": meta.get("iters", None),
        "DISTS(out, target)": d,
    }
    rows.append(row)
    outputs[method] = out_pil

df = pd.DataFrame(rows).sort_values(["time_sec", "DISTS(out, target)"])
df

C:\Users\mycos\anaconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\mycos\anaconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\mycos\anaconda3\lib\site-packages\DISTS_pytorch\DISTS_pt.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURI

,method,use_precond,solver,time_scope,time_sec,sinkhorn_iters,"DISTS(out, target)"
7,raw_variable_sinkhorn_lut,False,variable_sinkhorn,ot_only,0.413535,6.0,0.416578
2,precond_plain_sinkhorn_lut,True,plain_sinkhorn,ot_only,0.448854,3.0,0.427532
6,raw_plain_sinkhorn_lut,False,plain_sinkhorn,ot_only,0.454921,3.0,0.417146
3,precond_variable_sinkhorn_lut,True,variable_sinkhorn,ot_only,0.513196,5.0,0.428185
0,precond_emd_lut,True,emd,ot_only,2.482193,NaN,0.442454
4,raw_emd_lut,False,emd,ot_only,2.533187,NaN,0.441950
5,raw_pot_sinkhorn_lut,False,pot_sinkhorn,ot_only,12.435727,NaN,0.423548
1,precond_pot_sinkhorn_lut,True,pot_sinkhorn,ot_only,13.036814,NaN,0.422722
